# APAN 5200 — Logistic Regression Mock Quiz 2
**Dataset:** `spotify_data.csv`  
**Outcome:** `hit` = 1 if `rating >= 40`, else 0  
**Goal:** Predict whether a song qualifies as a 'hit'.

Same structure as original quiz: Part 1 = statsmodels | Part 2 = scikit-learn

## Variable Descriptions
- `track_duration`: Duration in ms
- `track_explicit`: Is explicit (0/1)
- `danceability`: Suitability for dancing (0–1)
- `energy`: Intensity & activity (0–1)
- `loudness`: Overall loudness in dB
- `mode`: Major (1) or Minor (0)
- `speechiness`: Presence of spoken words (0–1)
- `acousticness`: Acoustic confidence (0–1)
- `instrumentalness`: No-vocal likelihood (0–1)
- `liveness`: Audience presence (0–1)
- `valence`: Musical positiveness (0–1)
- `tempo`: BPM
- `time_signature`: Time signature
- `genre_*`: 10 binary genre dummies
- `hit`: **TARGET** — 1 if rating ≥ 40, 0 otherwise

---
## Part 1 — StatsModels

In [ ]:
import pandas as pd
import numpy as np
import math
import warnings
warnings.filterwarnings('ignore')
from statsmodels.formula.api import logit

In [ ]:
# ============================================================
# Q1: Read spotify_data.csv. Drop: id, performer, song, genre, key.
#     Create: hit = 1 if rating >= 40, else 0.
#     Split 70/30 using DataFrame.sample(), random_state=1031.
#     What % of songs in the TRAIN sample are hits?
# ANSWER: 44.02%
# ============================================================

data = pd.read_csv('spotify_data.csv')
data = data.drop(columns=['id','performer','song','genre','key'])
data['hit'] = (data['rating'] >= 40).astype(int)

train = data.sample(frac=0.7, random_state=1031)
test  = data.drop(labels=train.index)

pct_hit = round(train['hit'].mean() * 100, 2)
print(f'Train hit rate: {pct_hit}%')  # ✅ 44.02

In [ ]:
# ============================================================
# Q2: Compare hit vs non-hit songs on average ENERGY.
#     What is the average energy for HIT songs (hit=1)?
# ANSWER: 0.6356
# KEY INSIGHT: Hit songs have slightly higher energy.
# ============================================================

avg_energy = train.groupby('hit')['energy'].mean().round(4)
print(avg_energy)
print(f'Avg energy (hit=1): {round(train[train["hit"]==1]["energy"].mean(),4)}')  # ✅ 0.6356

In [ ]:
# ============================================================
# Q3: Fit model1 = logit('hit ~ energy'). Which conclusion?
# ANSWER: energy coefficient is POSITIVE (~0.83), p<0.05
#         → Higher energy songs are more likely to be hits
# ============================================================

model1 = logit('hit ~ energy', data=train).fit()
print(model1.summary())
print(f'\nenergy coeff: {round(model1.params["energy"],4)}')  # ✅ ~+0.83

In [ ]:
# ============================================================
# Q4: Based on model1, what is the predicted probability the
#     FIRST song in train is a hit?
# ANSWER: ~0.481
# ============================================================

first_song = train.iloc[[0]][['energy']]
prob_first = model1.predict(first_song)
print(f'First song energy = {round(train["energy"].iloc[0],4)}')
print(f'P(hit=1) = {round(prob_first.values[0],4)}')  # ✅ ~0.481

In [ ]:
# ============================================================
# Q5: Based on model1, what is P(hit | energy = 0.8)?
# ANSWER: ~0.477
# ============================================================

prob_e8 = model1.predict(pd.DataFrame({'energy':[0.8]}))
print(f'P(hit | energy=0.8) = {round(prob_e8.values[0],4)}')  # ✅ ~0.477

In [ ]:
# ============================================================
# Q6: If energy increases by 0.1, what is the % change in
#     likelihood of being a hit?
# ANSWER: +8.68%  (formula: (exp(β × 0.1) − 1) × 100)
# ============================================================

b_energy = model1.params['energy']
pct_change = (math.exp(b_energy * 0.1) - 1) * 100
print(f'energy coeff: {round(b_energy,4)}')
print(f'Per +0.1 energy, hit likelihood changes: {round(pct_change,2)}%')  # ✅ +8.68%

In [ ]:
# ============================================================
# Q7: Examine hit rate by GENRE (use the genre_ dummy variables).
#     Which genre has the HIGHEST hit rate?
# ANSWER: genre_dance (62.2%)
# ============================================================

genre_cols = [c for c in train.columns if c.startswith('genre_')]
genre_rates = {g: round(train[train[g]==1]['hit'].mean()*100,2) for g in genre_cols}
genre_rates_s = sorted(genre_rates.items(), key=lambda x: -x[1])
print('Hit rate by genre:')
for g,r in genre_rates_s:
    print(f'  {g}: {r}%')
# ✅ genre_dance: 62.2%  |  genre_blues: 28.46%  |  genre_space: 15.79%

In [ ]:
# ============================================================
# Q8: Fit model2 = logit('hit ~ genre_rock').
#     How does being a rock song affect likelihood of being a hit vs non-rock?
# ANSWER: +24.63%  (rock songs more likely to be hits, ~1.25x)
# ============================================================

model2 = logit('hit ~ genre_rock', data=train).fit()
print(model2.summary())
b_rock = model2.params['genre_rock']
print(f'\ngenre_rock coeff: {round(b_rock,4)}')
print(f'Pct change: {round((math.exp(b_rock)-1)*100,2)}%')  # ✅ +24.63%

In [ ]:
# ============================================================
# Q9: Fit model3 = logit('hit ~ valence'). Which conclusion?
# ANSWER: valence coefficient is NEGATIVE (~-0.53)
#         → More positive/happy songs are LESS likely to be hits
# ============================================================

model3 = logit('hit ~ valence', data=train).fit()
print(model3.summary())
print(f'\nvalence coeff: {round(model3.params["valence"],4)}')  # ✅ ~-0.53

In [ ]:
# ============================================================
# Q10: Fit model4 with ALL numeric audio features:
#      energy, valence, loudness, danceability, acousticness,
#      tempo, speechiness, track_duration, track_explicit, mode.
#      What is the Pseudo R² (McFadden)?
# ANSWER: run to compute
# ============================================================

model4 = logit(
    'hit ~ energy + valence + loudness + danceability + acousticness +'
    ' tempo + speechiness + track_duration + track_explicit + mode',
    data=train
).fit()
print(f'Pseudo R² (McFadden): {round(model4.prsquared,4)}')

In [ ]:
# ============================================================
# Q11: Based on model4, which variables SIGNIFICANTLY affect hit status?
# ============================================================

print(model4.summary())
sig = model4.pvalues[model4.pvalues < 0.05]
print('\nSignificant variables (p<0.05):')
print(sig)

In [ ]:
# ============================================================
# Q12: Based on model4, what is the impact of a ONE-UNIT increase
#      in VALENCE on the likelihood of being a hit?
# ANSWER: run to compute (valence is negative → higher valence decreases hit prob)
# ============================================================

b_valence = model4.params['valence']
impact = (math.exp(b_valence) - 1) * 100
print(f'valence coeff: {round(b_valence,4)}')
print(f'Per +1 unit valence, hit likelihood changes: {round(impact,2)}%')

---
## Part 2 — Scikit-Learn

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, recall_score, roc_auc_score

In [ ]:
# ============================================================
# Q13: Separate X and y (hit). Drop 'rating' from X.
#      Stratified 80/20 split, random_state=1031.
#      What % of the TRAIN sample are hits?
# ANSWER: 44.14%
# ============================================================

y = data['hit']
X = data.drop(columns=['hit', 'rating'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.8, random_state=1031, stratify=y
)

print(f'Train hit rate: {round(y_train.mean()*100,2)}%')  # ✅ 44.14

In [ ]:
# ============================================================
# Q14: What is the mean of 'track_explicit' in the train sample?
# ANSWER: 0.1204
# ============================================================

print(f'track_explicit mean (train): {round(X_train["track_explicit"].mean(),4)}')  # ✅ 0.1204

In [ ]:
# ============================================================
# Q15: Fit LogisticRegression(max_iter=10000, class_weight='balanced').
#      What is the predicted CLASS for the first observation in train?
# ANSWER: 0 (not a hit)  |  P(hit=1) ≈ 0.374
# NOTE: class_weight='balanced' handles the 56/44 class split.
# ============================================================

logit_sk = LogisticRegression(max_iter=10000, random_state=1031, class_weight='balanced')
logit_sk.fit(X_train, y_train)

y_pred_train       = logit_sk.predict(X_train)
y_pred_proba_train = logit_sk.predict_proba(X_train)
y_pred_test        = logit_sk.predict(X_test)
y_pred_proba_test  = logit_sk.predict_proba(X_test)

print(f'First obs predicted class : {y_pred_train[0]}')          # ✅ 0
print(f'First obs P(hit=1)        : {round(y_pred_proba_train[0,1],4)}')  # ✅ 0.374

In [ ]:
# ============================================================
# Q16: False Negatives are most costly (missing a hit song).
#      What is the Sensitivity in the TRAIN sample?
# ANSWER: 0.6426
# ============================================================

sens_train = recall_score(y_train, y_pred_train)
print(f'Sensitivity (train): {round(sens_train,4)}')  # ✅ 0.6426

cm = confusion_matrix(y_train, y_pred_train)
print(f'\nConfusion Matrix (train):\n{cm}')
print(f'TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}')

In [ ]:
# ============================================================
# Q17: What is the Sensitivity in the TEST sample?
# ANSWER: 0.6425
# NOTE: train ≈ test → excellent generalisation, no overfitting
# ============================================================

sens_test = recall_score(y_test, y_pred_test)
print(f'Sensitivity (test): {round(sens_test,4)}')  # ✅ 0.6425

cm_te = confusion_matrix(y_test, y_pred_test)
print(f'\nConfusion Matrix (test):\n{cm_te}')

In [ ]:
# ============================================================
# Q18: What is the AUC in the TRAIN sample?
# ANSWER: 0.6702
# ============================================================

auc_train = roc_auc_score(y_train, y_pred_proba_train[:,1])
print(f'AUC (train): {round(auc_train,4)}')  # ✅ 0.6702

In [ ]:
# ============================================================
# Q19: What is the AUC in the TEST sample?
# ANSWER: 0.6634
# KEY INSIGHT:
#   AUC slightly lower on test (~0.663 vs 0.670) — small, acceptable gap.
#   Like Mock 1, audio features alone are moderate predictors.
#   Compared to Mock 1 (popular threshold=50, AUC~0.68),
#   predicting a less extreme threshold (>=40) is similarly difficult.
# ============================================================

auc_test = roc_auc_score(y_test, y_pred_proba_test[:,1])
print(f'AUC (test): {round(auc_test,4)}')  # ✅ 0.6634

---
## Answer Summary

In [ ]:
summary = {
    'Q1  Train hit rate (%)':           '44.02',
    'Q2  Avg energy (hit=1)':            '0.6356',
    'Q3  energy effect':                 'Positive & significant (~+0.83)',
    'Q4  P(hit | first song)':           '~0.481',
    'Q5  P(hit | energy=0.8)':           '~0.477',
    'Q6  energy +0.1 → % change':        '+8.68%',
    'Q7  Highest hit rate genre':        'genre_dance (62.2%)',
    'Q8  genre_rock % change':           '+24.63%',
    'Q9  valence effect':                'Negative (~-0.53)',
    'Q10 Pseudo R² (model4)':            'run to compute',
    'Q11 Significant vars':              'run to compute',
    'Q12 valence +1 → % change':         'run to compute (negative)',
    'Q13 Stratified train rate (%)':     '44.14',
    'Q14 track_explicit mean (train)':   '0.1204',
    'Q15 First obs prediction':          '0 (not a hit)',
    'Q16 Sensitivity (train)':           '0.6426',
    'Q17 Sensitivity (test)':            '0.6425',
    'Q18 AUC (train)':                   '0.6702',
    'Q19 AUC (test)':                    '0.6634',
}
for k,v in summary.items():
    print(f'{k:<42} → {v}')